# GENIE vs production: νμ CC event rates (truth / mcnu only)

Compare **νμ CC** and **νμ CCQE** event rates between GENIE standalone flat ROOT and production MC truth (`mcnu` + `hdr` only).

No signal cut, no differential cross section, no reco — just truth counts and $E_\nu$ shapes.

- **GENIE (CC)**: `14_1000180400_CC_*.flat.root` — `PDGnu==14 & cc==1`.
- **GENIE (CCQE)**: same file with `Mode==1` (QE channel).
- **Production (CC)**: `(mc.iscc==1) & (mc.pdg==14)` on `mcnu`.
- **Production (CCQE)**: CC cut plus `mc.genie_mode==0`.
- **Rate** = $N / \mathrm{POT}$.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import pandas as pd
import numpy as np
import uproot
from os import path

%matplotlib inline
import matplotlib.pyplot as plt

import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana')
from pyanalib.split_df_helpers_new import dfs_from_dir
from analysis_village.numucc_1p0pi.utils import get_integrated_flux, get_active_volume
from analysis_village.numucc_1p0pi.constants import RHO, M_AR, N_A

import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)


## Configuration


In [ ]:
# GENIE standalone flat ROOT (CC νμ on Ar)
genie_file_dir = "/pnfs/sbnd/persistent/users/apapadop/GENIETweakedSamples/v3_6_2_AR23_20i_00_000_gen1_flux"
genie_filename = genie_file_dir + "/14_1000180400_CC_v3_6_2_AR23_20i_00_000.flat.root"
# genie_filename = "/pnfs/sbnd/scratch/users/sungbino/pac/xsec_charge/AR23_20i_00_000/gntp-numu-8.flat.root"

# Production MC truth (mcnu + hdr only)
mc_df_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC"
mc_filename_str = "sel_mup"


## Load GENIE flat ROOT


In [ ]:
events = uproot.open(genie_filename + ":FlatTree_VARS")
genie_arr = events.arrays(
    ["PDGnu", "cc", "Mode", "Enu_true", "fScaleFactor", "Weight"], library="np"
)

In [ ]:
genie_cc = (genie_arr["PDGnu"] == 14) & (genie_arr["cc"] == 1)
genie_enu = genie_arr["Enu_true"][genie_cc]
genie_w = (40.0 * genie_arr["fScaleFactor"] * genie_arr["Weight"])[genie_cc]

print(f"GENIE flat: {len(genie_arr['Enu_true']):,} entries, {genie_cc.sum():,} νμ CC")
print(f"  Enu range: {genie_enu.min():.3f} – {genie_enu.max():.3f} GeV")
print(f"  fScaleFactor unique: {np.unique(genie_arr['fScaleFactor']).size}")
print(f"  Weight unique: {np.unique(genie_arr['Weight']).size}")
print(f"  sum(weights) νμ CC: {genie_w.sum():.6e}")

## Load production MC (mcnu + hdr only)


In [ ]:
df_mc = dfs_from_dir(
    search_dir=mc_df_dir,
    filename_str=mc_filename_str,
    keys2load=["hdr", "mcnu"],
    n_max_concat=200,
)
mc_hdr_df = df_mc["hdr"]
mc_nu_df = df_mc["mcnu"] #.groupby(level=[0,1]).head(1)

# FV CUT!!
from makedf.util import InFV
mc_nu_df_FV = mc_nu_df[InFV(mc_nu_df.mc.position, det="SBND_nohighyz")]
mc_nu_df_FV = mc_nu_df_FV[np.abs(mc_nu_df_FV.mc.position.x) > 10]

mc_tot_pot = mc_hdr_df["pot"].sum()
print(f"Production: {len(mc_nu_df):,} mcnu rows")
print(f"  Total POT: {mc_tot_pot:.3e}")

mc_cc = (mc_nu_df["mc"]["iscc"] == 1) & (mc_nu_df["mc"]["pdg"] == 14)
mc_cc = mc_cc & mc_nu_df["mc"]["iscc"].notna() & mc_nu_df["mc"]["pdg"].notna()
prod_enu = np.asarray(mc_nu_df.loc[mc_cc, ("mc", "E")], dtype=float)

print(f"  νμ CC rows: {mc_cc.sum():,}")
print(f"  Enu range: {prod_enu.min():.3f} – {prod_enu.max():.3f} GeV")


In [ ]:
# mc_nu_df = mc_nu_df.groupby(level=[0,1]).head(1)


## νμ CC event rates


In [ ]:
n_genie_cc = int(genie_cc.sum())
n_prod_cc = int(mc_cc.sum())

rate_genie = n_genie_cc  # GENIE flat has no POT in file; count is per generated sample
rate_prod_per_pot = n_prod_cc / mc_tot_pot

print("=== νμ CC counts ===")
print(f"  GENIE flat (generated sample):  {n_genie_cc:,}")
print(f"  Production mcnu:                {n_prod_cc:,}")
print(f"  Ratio prod/GENIE:               {n_prod_cc / n_genie_cc:.4f}")
print()
print("=== Production rate ===")
print(f"  N_νμCC / POT = {rate_prod_per_pot:.6e} [/POT]")
print(f"  N_νμCC per 10^20 POT = {rate_prod_per_pot * 1e20:.4f}")
print()
print("GENIE flat file encodes exposure in fScaleFactor (constant per file).")
print(f"  GENIE sum(40×fScaleFactor×Weight) = {genie_w.sum():.6e}")


## $E_\nu$ distribution (νμ CC only, density-normalized)


In [ ]:
# comparison plot
bins = np.linspace(0, 4, 41)
centers = 0.5 * (bins[:-1] + bins[1:])

fig, ax = plt.subplots()
ax.hist(prod_enu, bins, histtype="step", density=True, lw=2, label="Production")
ax.hist(genie_enu, bins, histtype="step", density=True, lw=2, ls="--", label="GENIE")
ax.legend()

## $E_\nu$ vs production POT (counts per bin / POT)

Production rate spectrum: histogram of truth $E_\nu$ for νμ CC, divided by total MC POT.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
h_rate, _ = np.histogram(prod_enu, bins=bins)
h_rate = h_rate / mc_tot_pot
ax.step(centers, h_rate, where="mid", lw=2, label=r"Production $\nu_\mu$ CC / POT")
ax.set_xlabel(r"$E_\nu$ [GeV]")
ax.set_ylabel(r"$dN / dE_\nu$ [/POT/GeV]")
ax.set_title("Production νμ CC rate vs $E_\nu$ (truth mcnu)")
ax.legend()
fig.tight_layout()
plt.show()


## νμ CCQE event rates (closure)

Same comparison as above, restricted to **CCQE** interactions:

- **GENIE flat**: νμ CC with `Mode == 1` (QE channel in the flat tree; matches dedicated `*_CCQE_*.flat.root` samples).
- **Production**: νμ CC with `mc.genie_mode == 0` (QE in CAF / `categories.IsNuInFV_NumuCC_QE`).

In [ ]:
# CCQE subsets (reuse νμ CC masks from above)
genie_ccqe = genie_cc & (genie_arr["Mode"] == 1)
genie_enu_ccqe = genie_arr["Enu_true"][genie_ccqe]
genie_w_ccqe = (40.0 * genie_arr["fScaleFactor"] * genie_arr["Weight"])[genie_ccqe]

mc_ccqe = (
    mc_cc
    & (mc_nu_df["mc"]["genie_mode"] == 0)
    & mc_nu_df["mc"]["genie_mode"].notna()
)
prod_enu_ccqe = np.asarray(mc_nu_df.loc[mc_ccqe, ("mc", "E")], dtype=float)

n_genie_ccqe = int(genie_ccqe.sum())
n_prod_ccqe = int(mc_ccqe.sum())
rate_prod_ccqe_per_pot = n_prod_ccqe / mc_tot_pot

print("=== νμ CCQE counts ===")
print(f"  GENIE flat (Mode==1):           {n_genie_ccqe:,}  ({100*n_genie_ccqe/n_genie_cc:.1f}% of νμ CC)")
print(f"  Production (genie_mode==0):     {n_prod_ccqe:,}  ({100*n_prod_ccqe/n_prod_cc:.1f}% of νμ CC)")
print(f"  Ratio prod/GENIE:               {n_prod_ccqe / n_genie_ccqe:.4f}")
print()
print("=== Production CCQE rate ===")
print(f"  N_νμCCQE / POT = {rate_prod_ccqe_per_pot:.6e} [/POT]")
print(f"  N_νμCCQE per 10^20 POT = {rate_prod_ccqe_per_pot * 1e20:.4f}")
print()
print(f"  GENIE sum(40×fScaleFactor×Weight) CCQE = {genie_w_ccqe.sum():.6e}")

## $E_\nu$ distribution — νμ CCQE (density-normalized)

In [ ]:
fig, ax = plt.subplots()
ax.hist(prod_enu_ccqe, bins, histtype="step", density=True, lw=2, label="Production CCQE")
ax.hist(genie_enu_ccqe, bins, histtype="step", density=True, lw=2, ls="--", label="GENIE CCQE")
ax.set_xlabel(r"$E_\nu$ [GeV]")
ax.set_ylabel("Density")
ax.set_title(r"$\nu_\mu$ CCQE — truth $E_\nu$ shape")
ax.legend()
fig.tight_layout()
plt.show()

## $E_\nu$ vs production POT — νμ CCQE (counts per bin / POT)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
h_rate_ccqe, _ = np.histogram(prod_enu_ccqe, bins=bins)
h_rate_ccqe = h_rate_ccqe / mc_tot_pot
ax.step(centers, h_rate_ccqe, where="mid", lw=2, label=r"Production $\nu_\mu$ CCQE / POT")
ax.set_xlabel(r"$E_\nu$ [GeV]")
ax.set_ylabel(r"$dN / dE_\nu$ [/POT/GeV]")
ax.set_title("Production νμ CCQE rate vs $E_\\nu$ (truth mcnu)")
ax.legend()
fig.tight_layout()
plt.show()